This script calculates population exposed to high heat, based on yearly CDDs

- CDD data

from Climate Solutions Explorer (Werning et al. 2024): https://doi.org/10.5281/zenodo.14524342

Option to use single model or multi-model mean data

- Population data

New SSP data (2024 version) obtained from IMAGE Land

- Temperature time series

Used to interpolate CDD data per year, based on SSP5_H from scenariomip


In [59]:
import os
import xarray as xr
import numpy as np
import pandas as pd
import pym
import glob

**READ IN DATA**

In [73]:
def read_netcdf(filepath):
    dataset = xr.open_dataset(filepath)
    return dataset

ROOT_DIR = r'X:\user\crassierc\code\cooling\CoolingGap'
os.chdir(ROOT_DIR)

## EDIT INPUTS HERE ##
USE_MULTI_MODEL_MEAN = False
SINGLE_MODEL = 'ipsl-cm6a-lr'
SSP = 'ssp585'

# CDD DATA
if USE_MULTI_MODEL_MEAN:
    CDD_DIR = 'energy'
else: 
    CDD_DIR = 'intermediate_outputs_ensemble_members_energy'
correct_base_temps = ['18p3', '20p0', '24p0']
missing_base_temps = {
    '': '26p0'
}
gwl_labels = {
    '1p2': 1.2,
    '1p5': 1.5,
    '2p0': 2.0,
    '2p5': 2.5,
    '3p0': 3.0,
    '3p5': 3.5
}


def load_multi_model_mean(directory, gwl_labels, correct_base_temps, missing_base_temps):
    cdd_datasets = {}

    for base_temp in correct_base_temps:
        cdd_datasets[base_temp] = {}
        for gwl_label, gwl_value in gwl_labels.items():
            filename = f'cse_sdd_c_{base_temp}_ssp2_{gwl_label}_abs.nc4'
            filepath = os.path.join(directory, filename)
            print(f'  Loading: {os.path.basename(filepath)}')
            cdd_datasets[base_temp][gwl_value] = read_netcdf(filepath)

    for base_temp_value in missing_base_temps.values():
        cdd_datasets[base_temp_value] = {}
        for gwl_label, gwl_value in gwl_labels.items():
            filename = f'cse_sdd_c_ssp2_{gwl_label}_abs.nc4'
            filepath = os.path.join(directory, filename)
            print(f'  Loading: {os.path.basename(filepath)}')
            cdd_datasets[base_temp_value][gwl_value] = read_netcdf(filepath)

    return cdd_datasets

def find_single_model_file(directory, model_run, ssp, gwl_label, base_temp, file_type='abs'):
    folder_name = 'sdd' if base_temp in missing_base_temps.values() else f'sdd_{base_temp}'
    if base_temp in missing_base_temps.values():
        filename_pattern = f'{model_run}_{ssp}_{gwl_label}_sdd_c_global_*_{file_type}.nc4'
    else:
        filename_pattern = f'{model_run}_{ssp}_{gwl_label}_sdd_c_{base_temp}_global_*_{file_type}.nc4'
    pattern = os.path.join(directory, folder_name, filename_pattern)
    matches = glob.glob(pattern)
    if not matches:
        raise FileNotFoundError(f'No file found for pattern: {pattern}')
    if len(matches) > 1:
        print(f'  Warning: multiple matches found, using first: {matches}')
    return matches[0]

def load_single_model(directory, model_run, ssp, gwl_labels, correct_base_temps, missing_base_temps):
    cdd_datasets = {}

    for base_temp in correct_base_temps:
        cdd_datasets[base_temp] = {}
        for gwl_label, gwl_value in gwl_labels.items():
            filepath = find_single_model_file(directory, model_run, ssp, gwl_label, base_temp)
            print(f'  Loading: {os.path.basename(filepath)}')
            ds = read_netcdf(filepath)
            cdd_datasets[base_temp][gwl_value] = ds.rename({'sdd_c': f'sdd_c_{base_temp}'})

    for base_temp_value in missing_base_temps.values():
        cdd_datasets[base_temp_value] = {}
        for gwl_label, gwl_value in gwl_labels.items():
            filepath = find_single_model_file(directory, model_run, ssp, gwl_label, base_temp_value)
            print(f'  Loading: {os.path.basename(filepath)}')
            ds = read_netcdf(filepath)
            cdd_datasets[base_temp_value][gwl_value] = ds.rename({'sdd_c': f'sdd_c_{base_temp_value}'})

    return cdd_datasets

if USE_MULTI_MODEL_MEAN:
    print('Loading multi-model mean data')
    cdd_datasets = load_multi_model_mean(CDD_DIR, gwl_labels, correct_base_temps, missing_base_temps)
else:
    print(f'Loading single model data ({SINGLE_MODEL})')
    cdd_datasets = load_single_model(CDD_DIR, SINGLE_MODEL, SSP, gwl_labels, correct_base_temps, missing_base_temps)


# POPULATION DENSITY DATA (inhabitants / km2)
POP_DIR = 'Gridded_IMAGE_data/GPOPD'
ssps = ['SSP1', 'SSP2', 'SSP3', 'SSP5']
popd_datasets = {}

for ssp in ssps:
    filepath = os.path.join(POP_DIR, f'{ssp}', f'GPOPD_30MIN.NC')
    key = f'pop_{ssp}'
    popd_datasets[key] = read_netcdf(filepath)

# IMAGE REGIONS DATA
image_regions = read_netcdf(r'Gridded_IMAGE_data\GREG_30MIN.NC')

# AREA DATA (km2 / grid cell)
area = read_netcdf(r'Gridded_IMAGE_data\GAREACELLNOWATER_30MIN.NC')

# TEMPERATURE CHANGE TIME SERIES (TIMER OUTPUT)
gwl = pd.read_csv(r'GWL\GWL_SSP5_H.csv') 

Loading single model data (ipsl-cm6a-lr)
  Loading: ipsl-cm6a-lr_ssp585_1p2_sdd_c_18p3_global_1991_2021_abs.nc4
  Loading: ipsl-cm6a-lr_ssp585_1p5_sdd_c_18p3_global_2001_2031_abs.nc4
  Loading: ipsl-cm6a-lr_ssp585_2p0_sdd_c_18p3_global_2016_2046_abs.nc4
  Loading: ipsl-cm6a-lr_ssp585_2p5_sdd_c_18p3_global_2026_2056_abs.nc4
  Loading: ipsl-cm6a-lr_ssp585_3p0_sdd_c_18p3_global_2035_2065_abs.nc4
  Loading: ipsl-cm6a-lr_ssp585_3p5_sdd_c_18p3_global_2042_2072_abs.nc4
  Loading: ipsl-cm6a-lr_ssp585_1p2_sdd_c_20p0_global_1991_2021_abs.nc4
  Loading: ipsl-cm6a-lr_ssp585_1p5_sdd_c_20p0_global_2001_2031_abs.nc4
  Loading: ipsl-cm6a-lr_ssp585_2p0_sdd_c_20p0_global_2016_2046_abs.nc4
  Loading: ipsl-cm6a-lr_ssp585_2p5_sdd_c_20p0_global_2026_2056_abs.nc4
  Loading: ipsl-cm6a-lr_ssp585_3p0_sdd_c_20p0_global_2035_2065_abs.nc4
  Loading: ipsl-cm6a-lr_ssp585_3p5_sdd_c_20p0_global_2042_2072_abs.nc4
  Loading: ipsl-cm6a-lr_ssp585_1p2_sdd_c_24p0_global_1991_2021_abs.nc4
  Loading: ipsl-cm6a-lr_ssp585_1p5_s

**INTERPOLATE CDD DATA BETWEEN GLOBAL WARMING LEVELs**

In [74]:
def interpolate_cdd(datasets_for_base_temp, target_warming_level):

    gwls = sorted(datasets_for_base_temp.keys())
    if target_warming_level < min(gwls) or target_warming_level > max(gwls):
        raise ValueError(f"Target warming level {target_warming_level} is outside the available range "f"[{min(gwls)}, {max(gwls)}]")
    
    slices = []
    for gwl in gwls:
        dataset = datasets_for_base_temp[gwl]
        var_name = [v for v in dataset.data_vars][0]  # e.g. 'sdd_c_18p3'
        print(var_name)
        data = dataset[var_name].expand_dims({'gwl': [gwl]})
        slices.append(data)

    stacked = xr.concat(slices, dim='gwl')
    interpolated = stacked.interp(gwl=target_warming_level)
    interpolated = interpolated.assign_coords({
        'warming_level': target_warming_level
    })
    interpolated.attrs['description'] = f'Interpolated CDD values for {target_warming_level}°C warming level'
    
    return interpolated

**CALCULATE CDD PER YEAR FOR A TIME SERIES OF WARMING LEVELS**

In [75]:
def filter_gwl(datasets_for_base_temp, gwl_timeseries):

    gwls = sorted(datasets_for_base_temp.keys())
    min_threshold = min(gwls)
    max_threshold = max(gwls)

    valid_gwl_timeseries = gwl_timeseries[
        (gwl_timeseries['gwl'] >= min_threshold) & 
        (gwl_timeseries['gwl'] <= max_threshold)
    ].reset_index(drop=True)
    
    invalid_count = len(gwl_timeseries) - len(valid_gwl_timeseries)
    if invalid_count > 0:
        print(f"Filtered out {invalid_count} years with GWL values outside range [{min_threshold}, {max_threshold}]")
    
    return valid_gwl_timeseries


def interpolate_cdd_timeseries(datasets_for_base_temp, gwl_timeseries):

    interpolated_data = []
    
    for _, row in gwl_timeseries.iterrows():
        result = interpolate_cdd(datasets_for_base_temp, row['gwl'])
        result = result.assign_coords({'year': row['year']})
        interpolated_data.append(result)
 
    combined = xr.concat(interpolated_data, dim='year')
    combined = combined.assign_coords({
        'warming_level': ('year', [d.warming_level.item() for d in interpolated_data])
    })
    combined.attrs['description'] = 'Time series of interpolated CDD values'
    combined.attrs['year_range'] = f"{gwl_timeseries['year'].min()}-{gwl_timeseries['year'].max()}"
    
    return combined

In [76]:
# Gridded CDDs per year for different base temperatures, based on yearly GWL
interpolated_cdd_timeseries = {}

for base_temp, datasets_by_gwl in cdd_datasets.items():
    valid_gwl = filter_gwl(datasets_by_gwl, gwl)
    interpolated_cdd_timeseries[base_temp] = interpolate_cdd_timeseries(
        datasets_by_gwl, 
        valid_gwl
    )
    print(f"Completed time series interpolation for base temperature {base_temp}")

# Dataset for multiple warming levels (as coords) and base temps (as data variables)
combined_cdd_dataset_timeseries = xr.Dataset(
    {f'{temp}': data for temp, data in interpolated_cdd_timeseries.items()}
)

Filtered out 55 years with GWL values outside range [1.2, 3.5]
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sdd_c_18p3
sd

**IDENTIFY AREAS OF HEAT STRESS BASED ON YEARLY CDD THRESHOLD**

In [77]:
def get_cdd_threshold_masks(data, thresholds=[50, 100, 200, 400]):
    masks = {}
    for threshold in thresholds:
        mask_name = f'mask_{threshold}'
        masks[mask_name] = data >= threshold
    ds = xr.Dataset(masks)
    return ds

In [78]:
# Areas of heat stress per year, CDD threshold and base temperature
heatstress_masks = {}
for base_temp, data in combined_cdd_dataset_timeseries.items():
    heatstress_masks[base_temp] = get_cdd_threshold_masks(data)
    print(f"Created high heat masks for CDDs with base temperature {base_temp}")

Created high heat masks for CDDs with base temperature 18p3
Created high heat masks for CDDs with base temperature 20p0
Created high heat masks for CDDs with base temperature 24p0
Created high heat masks for CDDs with base temperature 26p0


In [79]:
# calculate population per grid cell from population density data
def calculate_population(gpopd_dataset, area_dataset, year):
    time = pd.Timestamp(f"{year}-01-01")
    pop_density = gpopd_dataset.GPOPD_30MIN.sel(time=time, method='nearest')
    area = area_dataset.GAREACELLNOWATER_30MIN.sel(time=time, method='nearest')
    population = pop_density * area
    return population

In [80]:
# Population data (pop / grid cell) per year and SSP
pop_datasets = {}
for ssp, dataset in popd_datasets.items():
    yearly_population = []
    years = pd.DatetimeIndex(dataset['time'].values).year.unique()
    for year in years:
        pop_data = calculate_population(dataset, area, year)
        yearly_population.append(pop_data.expand_dims(year=[year]))
    pop_datasets[ssp] = xr.concat(yearly_population, dim="year")
    print(f'added pop data for {ssp} to pop datasets')

added pop data for pop_SSP1 to pop datasets
added pop data for pop_SSP2 to pop datasets
added pop data for pop_SSP3 to pop datasets
added pop data for pop_SSP5 to pop datasets


In [81]:
# Interpolate population data between specified years for multiple SSP scenarios

def interpolate_population_timeseries(pop_datasets, interpolation_year=2026):
    interpolated_pop_data = {}
    for ssp, dataset in pop_datasets.items():
        dataset = dataset.assign_coords(year=dataset.year.values)
        interpolated = dataset.interp(year=interpolation_year)
        interpolated_pop_data[ssp] = interpolated
    combined = xr.Dataset(interpolated_pop_data)
    return combined

interpolated_pop_2026 = interpolate_population_timeseries(pop_datasets)

In [82]:
def add_interpolated_year_to_datasets(pop_datasets, interpolated_pop_dataset, interpolation_year=2026):

    updated_pop_datasets = {}
    for ssp in interpolated_pop_dataset.data_vars:
        original_dataset = pop_datasets[ssp]
        interpolated_data = interpolated_pop_dataset[ssp]
        
        existing_years = original_dataset.year.values
        insert_index = np.searchsorted(existing_years, interpolation_year)
        
        new_years = np.insert(existing_years, insert_index, interpolation_year)
        new_data = np.insert(
            original_dataset.values, 
            insert_index, 
            interpolated_data.values, 
            axis=0
        )
        
        updated_dataset = xr.DataArray(
            new_data, 
            coords={
                'year': new_years, 
                'latitude': original_dataset.latitude, 
                'longitude': original_dataset.longitude
            }, 
            dims=['year', 'latitude', 'longitude']
        )
        updated_pop_datasets[ssp] = updated_dataset
    
    return updated_pop_datasets

pop_datasets = add_interpolated_year_to_datasets(pop_datasets, interpolated_pop_2026)

In [83]:
def calculate_regional_population(population_data, region_data):

    pop_array = population_data.values
    reg_array = region_data.GREG_30MIN.isel(time=0).values
    
    regional_populations = {}
    
    for region_num in range(1, 27):
        mask = (reg_array == region_num)
        regional_pop = np.sum(pop_array[mask])
        regional_populations[region_num] = regional_pop
        
    return regional_populations

def get_regional_population(pop_dataset, region_dataset, year):
    
    pop_data_year = pop_dataset.sel(year=year)
    regional_pops = calculate_regional_population(pop_data_year, region_dataset)
    
    # Print results
    # print(f"\nRegional Populations for {year} :")
    # print("-" * 40)
    # total_pop = 0
    # for region, pop in regional_pops.items():
    #     pop_millions = pop / 1e6
    #     print(f"Region {region:1d}: {pop_millions:,.2f} million")
    #     total_pop += pop

    return regional_pops

In [84]:
# Calculate regional population per SSP and year

SSPs = [key for key in pop_datasets.keys()]

pop_data_dic = []

for ssp in SSPs:
    years = pop_datasets[ssp].coords['year'].to_numpy()    

    for year in years:
        try:
            reg_pop = get_regional_population(
                pop_datasets[ssp],
                image_regions,
                year
            )

            for key, value in reg_pop.items():
                pop_data_dic.append({
                    'SSP': ssp,
                    'Year': year,
                    'Region': key,
                    'RegionalPop': value
                })
        except Exception as e:
            print(f"Error adding data to pop_data_dic {ssp}, {year}: {e}")
    
    print(f"Processed pop data successfully for {ssp}")

regional_pop = pd.DataFrame(pop_data_dic)

Processed pop data successfully for pop_SSP1
Processed pop data successfully for pop_SSP2
Processed pop data successfully for pop_SSP3
Processed pop data successfully for pop_SSP5


**ESTIMATE POPULATION EXPOSED TO HEAT STRESS (PER REGION)**

In [85]:
def get_heatstress_pop(pop_data, region_data, heatstress_dataset, year, SSP, base_temp, mask_threshold):
 
    mask = heatstress_dataset[mask_threshold].sel(year=year).rename({'lon': 'longitude', 'lat': 'latitude'})
    pop_array = pop_data[SSP].sel(year=year).values
    reg_array = region_data.GREG_30MIN.isel(time=0).values
    mask_array = mask.values

    regional_heatstress_pop = {}
    regional_heatstress_pop_share = {}

    if not np.any(mask_array):
        print(f"Warning: Heat stress mask is empty for {year}, {base_temp}, {mask_threshold}")

    if np.all(np.isnan(pop_array)):
        print(f"Warning: Population data contains only NaN values for {year}, {SSP}")

    for region in range(1,27):
        region_mask = (reg_array == region)
        combined_mask = region_mask & mask_array
        total_region_pop = np.nansum(pop_array[region_mask])
        exposed_pop = np.nansum(pop_array[combined_mask])
        regional_heatstress_pop[region] = exposed_pop
        regional_heatstress_pop_share[region] = exposed_pop / total_region_pop
        # if exposed_pop > 0:
        #     print(f"% of region {region} pop exposed: {(regional_heatstress_pop_share):,.2f}%")

    return regional_heatstress_pop_share

In [86]:
common_years = sorted(set(pop_datasets[next(iter(pop_datasets))].year.values).intersection(
    *[heatstress_masks[base_temp].year.values for base_temp in heatstress_masks]
))
print(common_years)

SSPs = list(pop_datasets.keys())
base_temps = list(heatstress_masks.keys())
mask_thresholds = [var for var in heatstress_masks[next(iter(base_temps))].data_vars 
                  if var.startswith('mask_')]

pop_exposed_data = []

for base_temp in base_temps:
    heatstress_dataset = heatstress_masks[base_temp]
    if base_temp not in heatstress_masks:
        print(f"Error: base_temp '{base_temp}' is not in heatstress_masks")
        continue

    for mask_threshold in mask_thresholds:
        for ssp in SSPs:
            for year in common_years:
                try:
                    heatstress_pop = get_heatstress_pop(
                        pop_data=pop_datasets,
                        region_data=image_regions,
                        heatstress_dataset=heatstress_dataset,
                        year=year,
                        SSP=ssp,
                        base_temp=base_temp,
                        mask_threshold=mask_threshold
                    )

                    pop_exposed_data.extend([
                        {
                            'BaseTemp': base_temp,
                            'MaskThreshold': mask_threshold,
                            'SSP': ssp,
                            'Year': year,
                            'Region': region,
                            'ExposedPopulationShare': exposed_pop
                        } for region, exposed_pop in heatstress_pop.items()
                    ])
                    
                    # print(f"Processed: {base_temp}, {mask_threshold}, {ssp}, {year}")
                    
                except Exception as e:
                    print(f"Error processing {base_temp}, {mask_threshold}, {ssp}, {year}: {e}")

exposed_pop = pd.DataFrame(pop_exposed_data)

[2026, 2030, 2035, 2040, 2045, 2050, 2055, 2060, 2065, 2070, 2075, 2080, 2085, 2090, 2095, 2100]


**CALCULATE REGIONAL POPULATION-WEIGHTED CDDS**

In [87]:
def calculate_population_weighted_cdds(pop_dataset, cdd_dataset, region_dataset, year, base_temp):

    regions = region_dataset.GREG_30MIN.isel(time=0)
    cdd_selected = cdd_dataset[base_temp].sel(year=year)
    
    regions = regions.assign_coords(
        latitude=np.round(regions.latitude, 5),
        longitude=np.round(regions.longitude, 5)
    )
    population = pop_dataset.assign_coords(
        latitude=np.round(pop_dataset.latitude, 5),
        longitude=np.round(pop_dataset.longitude, 5)
    )

    cdd_regridded = cdd_selected.interp(
        lat=population.latitude,
        lon=population.longitude
    )

    regional_weighted_cdds = {}
    
    unique_regions = np.unique(regions.values[~np.isnan(regions.values)])
    
    for region in unique_regions:
        region_mask = (regions == region)
        
        region_pop = population.where(region_mask)
        region_cdds = cdd_regridded.where(region_mask)
        
        total_pop = region_pop.sum().values
        
        if total_pop > 0:
            weighted_cdds = (
                (region_pop * region_cdds).sum().values / total_pop
            )
            regional_weighted_cdds[int(region)] = float(weighted_cdds)
        else:
            regional_weighted_cdds[int(region)] = 0.0
    
    return regional_weighted_cdds

In [88]:
years = [2026, 2030, 2040, 2050, 2060, 2070, 2080, 2090, 2100]

combined_cdd_gwl_data = []

for base_temp in base_temps:
    for ssp in SSPs:
        for year in years:
            try:
                weighted_cdds = calculate_population_weighted_cdds(
                    pop_dataset=pop_datasets[ssp],
                    cdd_dataset=combined_cdd_dataset_timeseries,
                    region_dataset=image_regions,
                    year=year,
                    base_temp = base_temp
                )
                
                combined_cdd_gwl_data.extend([
                    {
                        'SSP': ssp,
                        'Year': year,
                        'BaseTemp': base_temp,
                        'Region': region,
                        'weighted_cdd': cdd_value,
                        'warming_level': float(combined_cdd_dataset_timeseries.warming_level.sel(year=year).values)
                    } for region, cdd_value in weighted_cdds.items()
                ])

                print(f"Processed successfully for {base_temp}, {ssp}, {year}")

            except Exception as e:
                print(f"Error processing {base_temp}, {ssp}, {year}: {str(e)}")

pop_weighted_cdd = pd.DataFrame(combined_cdd_gwl_data)

Processed successfully for 18p3, pop_SSP1, 2026
Processed successfully for 18p3, pop_SSP1, 2030
Processed successfully for 18p3, pop_SSP1, 2040
Processed successfully for 18p3, pop_SSP1, 2050
Processed successfully for 18p3, pop_SSP1, 2060
Processed successfully for 18p3, pop_SSP1, 2070
Processed successfully for 18p3, pop_SSP1, 2080
Processed successfully for 18p3, pop_SSP1, 2090
Processed successfully for 18p3, pop_SSP1, 2100
Processed successfully for 18p3, pop_SSP2, 2026
Processed successfully for 18p3, pop_SSP2, 2030
Processed successfully for 18p3, pop_SSP2, 2040
Processed successfully for 18p3, pop_SSP2, 2050
Processed successfully for 18p3, pop_SSP2, 2060
Processed successfully for 18p3, pop_SSP2, 2070
Processed successfully for 18p3, pop_SSP2, 2080
Processed successfully for 18p3, pop_SSP2, 2090
Processed successfully for 18p3, pop_SSP2, 2100
Processed successfully for 18p3, pop_SSP3, 2026
Processed successfully for 18p3, pop_SSP3, 2030
Processed successfully for 18p3, pop_SSP

**MERGE HEATSTRESS ESTIMATE WITH REGIONAL CDDs**

In [89]:
# Extract common years between exposed population and cdd data and filter
cdd_years = pop_weighted_cdd['Year'].unique()
exposed_years = exposed_pop['Year'].unique()
common_years = np.intersect1d(cdd_years, exposed_years)
print(common_years) # TODO: check if same as other common_years. if so delete

columns = ["BaseTemp", "Region", "SSP", "MaskThreshold", "warming_level"]
exposed_pop_cdd = (exposed_pop[exposed_pop['Year'].isin(common_years)]
                   .merge(
                       pop_weighted_cdd[pop_weighted_cdd['Year'].isin(common_years)]
                       [['BaseTemp', 'SSP', 'Year', 'Region', 'warming_level']],
                        on=['BaseTemp', 'SSP', 'Year', 'Region'],
                        how='left'
                    )
                    .merge(
                        regional_pop,
                        on=['SSP', 'Year', 'Region']
                    ).set_index(columns)
                    .dropna())

exposed_pop_series = exposed_pop_cdd['ExposedPopulationShare']

[2026 2030 2040 2050 2060 2070 2080 2090 2100]


**Create mym file for TIMER**

In [90]:
def write_mym(df, output_dir, value_column='ExposedPopulationShare', year_column='Year', base_temp='24p0'):
    
    os.makedirs(output_dir, exist_ok=True)

    ssp_list = df.index.get_level_values("SSP").unique()
    mask_order = df.index.get_level_values("MaskThreshold").unique().tolist()
    warming_order = df.index.get_level_values("warming_level").unique().tolist()
    
    for ssp in ssp_list:
        try:
            data_array = (
                df
                .xs(
                    (ssp, base_temp),
                    level=["SSP", "BaseTemp"],
                )
                .to_xarray()
            )

            data_array = data_array.reindex({"Region": range(1, 27)})
            df_pym = data_array.to_series()
            df_pym = df_pym.reorder_levels(["Region", "MaskThreshold", "warming_level"])
            
            regions = range(1, 27)
            new_index = pd.MultiIndex.from_product([regions, mask_order, warming_order], names=["Region", "MaskThreshold", "warming_level"])
            df_pym = df_pym.reindex(new_index)
            print(df_pym)

            ssp_id = ssp[4:9]
            filename = f'heat_exposed_pop_share_{ssp_id}.dat'
            filepath = os.path.join(output_dir, filename)
            pym.write_mym(data=df_pym, filename=filepath, variable_name=value_column)
            print(f"Saved {filepath}")
            
        except Exception as e:
            print(f"Error processing {ssp}: {str(e)}")
            import traceback
            traceback.print_exc()
            
    return df_pym

output_dir = os.path.join(ROOT_DIR, 'EXPOSED_POP_FOR_TIMER_IPSL_19_03_26')
final = write_mym(exposed_pop_series, output_dir)

Region  MaskThreshold  warming_level
1       mask_50        1.214875         0.000000
                       1.311853         0.000000
                       1.573306         0.000000
                       1.867270         0.019295
                       2.175638         0.066769
                                          ...   
26      mask_400       2.175638         0.434105
                       2.480449         0.483161
                       2.783638         0.562151
                       3.083198         0.622684
                       3.373593         0.688694
Name: ExposedPopulationShare, Length: 936, dtype: float32
Saved X:\user\crassierc\code\cooling\CoolingGap\EXPOSED_POP_FOR_TIMER_IPSL_NEW_19_03_26\heat_exposed_pop_share_SSP1.dat
Region  MaskThreshold  warming_level
1       mask_50        1.214875         0.000000
                       1.311853         0.000000
                       1.573306         0.000000
                       1.867270         0.019340
             

Create lookup table for timer-prism

In [91]:
# import os
# import pandas as pd

# def write_prism(df, output_dir, value_column='ExposedPopulationShare', base_temp='cdd_24p0'):
#     os.makedirs(output_dir, exist_ok=True)

#     ssp_list = df.index.get_level_values("SSP").unique()
#     mask_order = df.index.get_level_values("MaskThreshold").unique().tolist()
#     warming_order = df.index.get_level_values("warming_level").unique().tolist()

#     warming_order_sorted = sorted(warming_order, key=lambda x: float(x))
#     regions = list(range(1, 27))

#     for ssp in ssp_list:
#         try:
#             data_array = (
#                 df
#                 .xs((ssp, base_temp), level=["SSP", "BaseTemp"])
#                 .to_xarray()
#             )

#             data_array = data_array.reindex({
#                 "Region": regions,
#                 "MaskThreshold": mask_order,
#                 "warming_level": warming_order_sorted,
#             })

#             series = (
#                 data_array
#                 .to_series()
#                 .reorder_levels(["Region", "MaskThreshold", "warming_level"])
#                 .sort_index()
#             )

#             ssp_id = ssp[4:9]
#             filename = f'heat_exposed_pop_share_{ssp_id}.dat'
#             filepath = os.path.join(output_dir, filename)

#             with open(filepath, "w") as f:
#                 # f.write(f"real {value_column}[26, {len(mask_order)}](TemperatureEST) =\n")

#                 # for warming in warming_order_sorted:
#                 #     for region in regions:
#                 #         row_vals = []
#                 #         for mask in mask_order:
#                 #             values = series.get((region, mask, warming), float("nan"))
#                 #             if pd.isna(values):
#                 #                 values = 0.0
#                 #             row_vals.append(f"{values:.6f}")
#                 #         f.write(f"{warming}\t" + "\t".join(row_vals) + "\n")

#                 # f.write(";\n")

#                 f.write(f"real {value_column}[26, {len(mask_order)}](TemperatureEST) =\n")

#                 for warming in warming_order_sorted:
#                     f.write(f"{warming},\n")

#                     for region in regions:
#                         row_vals = []
#                         for mask in mask_order:
#                             value = series.get((region, mask, warming), float("nan"))
#                             if pd.isna(value):
#                                 value = 0.0
#                             row_vals.append(f"{value:.6f},")
#                         f.write("".join(row_vals) + "\n")  # Remove final comma just in case

#                 f.write(";\n")

#             print(f"Saved {filepath}")

#         except Exception as e:
#             print(f"Error processing {ssp}: {str(e)}")
#             import traceback
#             traceback.print_exc()

#     return None


# output_dir = os.path.join(ROOT_DIR, 'EXPOSED_POP_FOR_TIMER_PRISM_test')
# final = write_prism(exposed_pop_series, output_dir)